In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 — Bronze Stream Ingest
# MAGIC Auto Loader picks up whatever JSON files are sitting in the landing folder,
# MAGIC stamps them with ingestion metadata, writes to Bronze. `trigger(availableNow=True)`
# MAGIC processes everything currently there, then stops — a stream that behaves like a
# MAGIC batch job so it's verifiable with exact numbers, same as every other project.

# COMMAND ----------

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

# COMMAND ----------

import importlib
import iot_pipeline.config
importlib.reload(iot_pipeline.config)
from iot_pipeline.config import BRONZE_TABLE, LANDING_PATH, CHECKPOINT_BASE
from iot_pipeline.transforms import add_ingest_metadata

# COMMAND ----------

checkpoint_path = CHECKPOINT_BASE + "bronze/"

raw_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .load(LANDING_PATH))

bronze_stream = add_ingest_metadata(raw_stream)

query = (bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .table(BRONZE_TABLE))

query.awaitTermination()

# COMMAND ----------

# MAGIC %md ## Validation

# COMMAND ----------

count = spark.table(BRONZE_TABLE).count()
assert count == 1200, f"Expected 1,200 rows, got {count}"
print(f"✅ {BRONZE_TABLE}: {count:,} rows")
print("Re-running this notebook won't reprocess or duplicate rows — Auto Loader's checkpoint remembers what it already ingested.")